# Alpha-weighted collective-choice simulation

This notebook is self-contained. It first validates the empirical CCEI distribution in `data/panel_individual.dta`, estimates the person-wave CRRA distribution, calibrates choice precision to empirical CCEI quantiles, and runs the alpha-weighted collective-choice simulation.

## Functions

In [1]:
"""Empirically calibrated alpha-weighted collective-choice simulation.

This script implements the investigations requested for Section 5.4:

1. Estimate individual EUT-CRRA curvature by person-wave NLLS, adapting the
   Choi-Fisman-Gale-Kariv (AER 2007) piecewise demand specification while
   fixing disappointment aversion at one.
2. Build small/medium/large preference-heterogeneity scenarios from the
   full empirical distribution of the estimated CRRA parameter.
3. Calibrate logit precision to Q25, Q50, and Q75 of the empirical CCEI
   distributions for both individual and collective choices.
4. Simulate 18 choices separately for member i, member j, and the group by
   independently sampling budgets from the full experimental budget pool.
5. Compare the Pareto weight alpha with 1 - Ihat_ig under both the original
   CRRA cardinalization and a positive-affine reference normalization, and
   invert the simulated curves to report rough alphas corresponding to the
   observed experimental revealed-preference distance.

The default run is intentionally publication-scale and can take a long time.
Use ``--smoke-test`` first.  Every substantive convention is collected in
DEFAULT_CONFIG and written to the output directory with the results.
"""

from __future__ import annotations

import argparse
import json
import math
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.optimize import minimize


SCRIPT_DIR = Path.cwd().resolve()
DEFAULT_DATA_DIR = SCRIPT_DIR / "data"
DEFAULT_OUTPUT_DIR = SCRIPT_DIR / "results" / "alpha_weighted_simulation"


@dataclass
class SimulationConfig:
    data_dir: str = str(DEFAULT_DATA_DIR)
    output_dir: str = str(DEFAULT_OUTPUT_DIR)
    base_raw_file: str = "base_raw.dta"
    end_raw_file: str = "end_raw.dta"
    panel_individual_file: str = "panel_individual.dta"

    # Experimental design
    n_budgets: int = 18
    n_options: int = 200
    budget_pool: str = "all"  # all or rational_legacy
    legacy_budget_file: str = "baseline_raw_129groups.csv"
    sample_budgets_with_replacement: bool = False

    # AER-2007 constrained NLLS
    boundary_w: float = 0.001
    # None uses every successful person-wave estimate.  A numeric value keeps
    # only estimates whose individual CCEI is at least that threshold.
    nlls_main_min_ccei: float | None = None
    min_nlls_observations: int = 12

    # Empirical definitions and rho scenarios
    high_ccei_threshold: float = 0.90
    heterogeneity_quantiles: dict[str, tuple[float, float]] = field(
        default_factory=lambda: {
            "small": (0.40, 0.60),
            "medium": (0.25, 0.75),
            "large": (0.10, 0.90),
        }
    )

    # Choice-error calibration
    gamma_grid: list[float] = field(
        # A wide range is necessary under the unnormalized CRRA utility.  For
        # large rho, utility differences can be extremely small, so gamma may
        # need to be much larger than 500 to generate high CCEI.
        default_factory=lambda: list(np.geomspace(1e-6, 1e14, 61))
    )
    gamma_calibration_replications: int = 150
    gamma_calibration_tolerance: float = 0.03
    gamma_group_alpha_reference: float = 0.50

    # Main alpha simulation
    alpha_grid: list[float] = field(
        default_factory=lambda: list(np.round(np.arange(0.0, 1.01, 0.10), 10))
    )
    simulation_replications: int = 200
    noise_scenarios: list[str] = field(
        default_factory=lambda: ["high_high", "high_low", "low_low"]
    )
    # Vary group choice precision separately instead of mechanically tying it
    # to the two members' precision types.
    group_precision_scenarios: list[str] = field(
        default_factory=lambda: ["low", "median", "high"]
    )
    utility_normalizations: list[str] = field(default_factory=lambda: ["raw"])
    reference_consumption: float | None = None

    # Reproducibility and computation
    master_seed: int = 20260823
    n_jobs: int = -1
    backend: str = "loky"
    verbose: int = 5


DEFAULT_CONFIG = SimulationConfig()


# ---------------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------------


def id_as_string(values: pd.Series) -> pd.Series:
    return values.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


def seed_rng(master_seed: int, *keys: int) -> np.random.Generator:
    return np.random.default_rng(np.random.SeedSequence([master_seed, *keys]))


def json_ready(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_ready(v) for v in value]
    return value


def save_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(json_ready(obj), indent=2), encoding="utf-8")


def ensure_required_columns(df: pd.DataFrame, columns: Iterable[str], label: str) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


# ---------------------------------------------------------------------------
# CCEI and cross-partition revealed-preference calculations
# ---------------------------------------------------------------------------


def revealed_relations(p: np.ndarray, x: np.ndarray, e: float) -> tuple[np.ndarray, np.ndarray]:
    costs = p.T @ x
    expenditure = np.diag(costs)
    return (e * expenditure)[:, None] >= costs, (e * expenditure)[:, None] > costs


def transitive_closure(r0: np.ndarray) -> np.ndarray:
    r = r0.copy()
    for k in range(r.shape[0]):
        r |= r[:, [k]] & r[[k], :]
    return r


def garp(p: np.ndarray, x: np.ndarray, e: float) -> int:
    r0, p0 = revealed_relations(p, x, e)
    r = transitive_closure(r0)
    return int(not np.any(r & p0.T))


def ccei_garp(p: np.ndarray, x: np.ndarray, tol: float = 1e-10) -> float:
    if garp(p, x, 1.0):
        return 1.0
    low, high = 0.0, 1.0
    while high - low > tol:
        mid = (low + high) / 2
        if garp(p, x, mid):
            low = mid
        else:
            high = mid
    return low


def ccei_dataset(data: dict[str, np.ndarray]) -> float:
    p = np.vstack((1.0 / data["intercept_x"], 1.0 / data["intercept_y"]))
    x = np.vstack((data["coord_x"], data["coord_y"]))
    return ccei_garp(p, x)


def cross_garp(p: np.ndarray, x: np.ndarray, side: np.ndarray, e: float) -> int:
    r0, p0 = revealed_relations(p, x, e)
    r = transitive_closure(r0)
    strict_on_cycle = p0 & r.T
    for a in np.flatnonzero(strict_on_cycle.any(axis=1)):
        component = (r[a, :] & r[:, a]).copy()
        component[a] = True
        component |= strict_on_cycle[a, :]
        if np.any(side[component] == 0) and np.any(side[component] == 1):
            return 0
    return 1


def ex_cross(p: np.ndarray, x: np.ndarray, side: np.ndarray, tol: float = 1e-6) -> float:
    if cross_garp(p, x, side, 1.0):
        return 1.0
    low, high, estar = 0.0, 1.0, 0.0
    while high - low > tol:
        mid = (low + high) / 2
        if cross_garp(p, x, side, mid):
            estar = low = mid
        else:
            high = mid

    costs = p.T @ x
    expenditure = np.diag(costs)
    ratios = costs / expenditure[:, None]
    mask = (~np.eye(len(side), dtype=bool)) & (ratios < 1 - 1e-12)
    hi = float(np.max(ratios[mask], initial=0.0))
    return 1.0 if estar > hi + 1e-9 else estar


def ex_from_datasets(individual: dict[str, np.ndarray], group: dict[str, np.ndarray]) -> float:
    ix = np.r_[individual["intercept_x"], group["intercept_x"]]
    iy = np.r_[individual["intercept_y"], group["intercept_y"]]
    cx = np.r_[individual["coord_x"], group["coord_x"]]
    cy = np.r_[individual["coord_y"], group["coord_y"]]
    p = np.vstack((1.0 / ix, 1.0 / iy))
    x = np.vstack((cx, cy))
    side = np.r_[
        np.zeros(len(individual["coord_x"]), dtype=np.int8),
        np.ones(len(group["coord_x"]), dtype=np.int8),
    ]
    return ex_cross(p, x, side)


def stack_individual_datasets(*datasets: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    keys = ("intercept_x", "intercept_y", "coord_x", "coord_y")
    return {key: np.concatenate([data[key] for data in datasets]) for key in keys}


def ihat_from_ex(ex_i: float, ex_j: float, ex_ij: float, tol: float = 1e-6) -> tuple[float, float]:
    c_i, c_j, c_ij = 1 - ex_i, 1 - ex_j, 1 - ex_ij
    if not np.isfinite(c_ij) or c_ij <= tol:
        return np.nan, np.nan
    ihat_i = 0.5 + (c_i - c_j) / (2 * c_ij)
    if ihat_i < -1e-7 or ihat_i > 1 + 1e-7:
        raise AssertionError(f"Ihat outside [0,1]: {ihat_i}")
    ihat_i = float(np.clip(ihat_i, 0.0, 1.0))
    return ihat_i, 1.0 - ihat_i


# ---------------------------------------------------------------------------
# Data loading and empirical samples
# ---------------------------------------------------------------------------


def prepare_raw(path: Path, post: int) -> pd.DataFrame:
    raw = pd.read_stata(path, convert_categoricals=False)
    ensure_required_columns(
        raw,
        [
            "id", "partner_id", "group_id", "round_number", "mover",
            "coord_x", "coord_y", "intercept_x", "intercept_y",
        ],
        path.name,
    )
    raw = raw.copy()
    for col in ("id", "partner_id", "group_id"):
        raw[col] = id_as_string(raw[col])
    for col in ("round_number", "mover"):
        raw[col] = pd.to_numeric(raw[col], errors="coerce").astype("Int64")
    for col in ("coord_x", "coord_y", "intercept_x", "intercept_y"):
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    raw["post"] = int(post)
    return raw


def load_panel_individual(path: Path) -> pd.DataFrame:
    panel = pd.read_stata(path, convert_categoricals=False)
    ensure_required_columns(
        panel,
        ["group_id", "id", "post", "ccei_i", "ccei_g"],
        path.name,
    )
    panel = panel.copy()
    panel["group_id"] = id_as_string(panel["group_id"])
    panel["id"] = id_as_string(panel["id"])
    panel["post"] = pd.to_numeric(panel["post"], errors="raise").astype(int)
    return panel


def canonical_budget_pool(raw_frames: list[pd.DataFrame]) -> pd.DataFrame:
    pieces = []
    for raw in raw_frames:
        individual = raw.loc[raw["round_number"].between(1, 18)].copy()
        group = raw.loc[raw["round_number"].between(19, 36) & (raw["mover"] == 1)].copy()
        pieces.extend([individual, group])
    pool = pd.concat(pieces, ignore_index=True)
    pool = pool.dropna(subset=["intercept_x", "intercept_y"])
    pool = pool.loc[(pool["intercept_x"] > 0) & (pool["intercept_y"] > 0)].copy()
    pool["budget_source"] = np.where(pool["round_number"] <= 18, "individual", "group")
    return pool[["intercept_x", "intercept_y", "post", "budget_source"]].reset_index(drop=True)


def legacy_rational_budget_pool(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path)
    ensure_required_columns(raw, ["intercept_x", "intercept_y"], path.name)
    if {"new2_I_ig", "new2_I_jg"}.issubset(raw.columns):
        raw = raw.loc[raw["new2_I_ig"].notna() & raw["new2_I_jg"].notna()].copy()
    raw = raw.loc[(raw["intercept_x"] > 0) & (raw["intercept_y"] > 0)].copy()
    raw["post"] = 0
    raw["budget_source"] = "legacy_rational"
    return raw[["intercept_x", "intercept_y", "post", "budget_source"]].reset_index(drop=True)


def empirical_ccei_targets(panel: pd.DataFrame, threshold: float) -> dict[str, float]:
    people = panel.drop_duplicates(["id", "post"])[["id", "post", "ccei_i"]].dropna()
    groups = panel.drop_duplicates(["group_id", "post"])[["group_id", "post", "ccei_g"]].dropna()
    individual_quantiles = people["ccei_i"].quantile([0.25, 0.50, 0.75])
    group_quantiles = groups["ccei_g"].quantile([0.25, 0.50, 0.75])
    return {
        # Professor's requested calibration points: CCEI Q25, median, Q75.
        # The threshold is retained only for legacy descriptive outputs.
        "legacy_high_ccei_threshold": threshold,
        "individual_low": float(individual_quantiles.loc[0.25]),
        "individual_median": float(individual_quantiles.loc[0.50]),
        "individual_high": float(individual_quantiles.loc[0.75]),
        "group_low": float(group_quantiles.loc[0.25]),
        "group_median": float(group_quantiles.loc[0.50]),
        "group_high": float(group_quantiles.loc[0.75]),
        "individual_overall_mean": float(people["ccei_i"].mean()),
        "group_overall_mean": float(groups["ccei_g"].mean()),
        "n_individual": int(len(people)),
        "n_group": int(len(groups)),
    }


# ---------------------------------------------------------------------------
# EUT-CRRA NLLS: AER-2007 specification with disappointment alpha fixed at 1
# ---------------------------------------------------------------------------


def adjust_boundary_choices(
    coord_x: np.ndarray,
    coord_y: np.ndarray,
    intercept_x: np.ndarray,
    intercept_y: np.ndarray,
    w: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Replicate the AER NLLS boundary adjustment for the demand ratio x/y."""
    mx = coord_x.astype(float).copy()
    my = coord_y.astype(float).copy()
    x_tiny = coord_x < w * coord_y
    y_tiny = coord_y < w * coord_x
    mx[x_tiny] = w * intercept_y[x_tiny]
    my[y_tiny] = w * intercept_x[y_tiny]
    both_tiny = x_tiny & y_tiny
    mx[both_tiny] = w * intercept_y[both_tiny]
    my[both_tiny] = w * intercept_x[both_tiny]
    return mx, my


def eut_crra_predicted_log_demand_ratio(log_price_ratio: np.ndarray, rho: float, w: float) -> np.ndarray:
    # With disappointment alpha=1, the five-case AER demand function collapses
    # to the EUT interior FOC, censored at the two adjusted boundaries.
    boundary = -math.log(w)
    return np.clip(-log_price_ratio / rho, -boundary, boundary)


def estimate_one_crra_nlls(
    sub: pd.DataFrame,
    w: float,
    min_observations: int,
) -> dict[str, Any]:
    required = ["coord_x", "coord_y", "intercept_x", "intercept_y"]
    clean = sub.dropna(subset=required).copy()
    clean = clean.loc[(clean["intercept_x"] > 0) & (clean["intercept_y"] > 0)]
    if len(clean) < min_observations:
        return {"rho_hat": np.nan, "ssr": np.nan, "r2_nlls": np.nan, "n_obs": len(clean), "success": False}

    cx = clean["coord_x"].to_numpy(float)
    cy = clean["coord_y"].to_numpy(float)
    ix = clean["intercept_x"].to_numpy(float)
    iy = clean["intercept_y"].to_numpy(float)
    mx, my = adjust_boundary_choices(cx, cy, ix, iy, w)
    if np.any(mx <= 0) or np.any(my <= 0):
        return {"rho_hat": np.nan, "ssr": np.nan, "r2_nlls": np.nan, "n_obs": len(clean), "success": False}

    log_demand_ratio = np.log(mx / my)
    # Prices are (1/intercept_x, 1/intercept_y), hence p_x/p_y = iy/ix.
    log_price_ratio = np.log(iy / ix)

    def objective(log_rho_array: np.ndarray) -> float:
        log_rho = float(np.asarray(log_rho_array).reshape(-1)[0])
        # Match the AER MATLAB code: rho=exp(pm(2)), with only a tiny
        # numerical floor h=1e-17 and no finite upper bound.
        rho = float(np.exp(log_rho)) if log_rho < 709.0 else np.inf
        rho = max(rho, 1e-17)
        fitted = eut_crra_predicted_log_demand_ratio(log_price_ratio, rho, w)
        return float(np.sum((log_demand_ratio - fitted) ** 2))

    # The original replication code uses pm0(2)=-2 and unconstrained
    # fminsearch. SciPy's Nelder-Mead is the corresponding implementation.
    opt = minimize(
        objective,
        x0=np.array([-2.0]),
        method="Nelder-Mead",
        options={"xatol": 1e-10, "fatol": 1e-10, "maxiter": 10_000},
    )
    log_rho_hat = float(opt.x[0])
    rho_hat = float(np.exp(log_rho_hat)) if log_rho_hat < 709.0 else np.inf
    rho_hat = max(rho_hat, 1e-17)
    ssr = float(opt.fun)
    sst = float(np.sum(log_demand_ratio**2))  # same zero-centered SST as the AER code
    r2 = np.nan if sst <= 1e-14 else 1.0 - ssr / sst
    return {
        "rho_hat": rho_hat,
        "log_rho_hat": log_rho_hat,
        "ssr": ssr,
        "r2_nlls": r2,
        "n_obs": len(clean),
        "success": bool(opt.success),
        "optimizer_iterations": int(opt.nit),
        "optimizer_message": str(opt.message),
    }


def estimate_crra_distribution(
    raw_frames: list[pd.DataFrame],
    panel: pd.DataFrame,
    cfg: SimulationConfig,
) -> pd.DataFrame:
    analysis_people = panel.drop_duplicates(["id", "post"])[["id", "post", "ccei_i"]]
    individual_raw = pd.concat(
        [raw.loc[raw["round_number"].between(1, 18)].copy() for raw in raw_frames],
        ignore_index=True,
    )
    individual_raw = individual_raw.merge(analysis_people, on=["id", "post"], how="inner", validate="many_to_one")

    rows: list[dict[str, Any]] = []
    for (person_id, post), sub in individual_raw.groupby(["id", "post"], sort=True):
        result = estimate_one_crra_nlls(
            sub,
            w=cfg.boundary_w,
            min_observations=cfg.min_nlls_observations,
        )
        result.update(
            {
                "id": person_id,
                "post": int(post),
                "ccei_i": float(sub["ccei_i"].iloc[0]),
                "estimation_unit": "person_wave_18",
            }
        )
        rows.append(result)

    estimates = pd.DataFrame(rows)
    estimates = apply_main_calibration_sample(estimates, cfg)
    return estimates.sort_values(["id", "post"]).reset_index(drop=True)


def apply_main_calibration_sample(
    estimates: pd.DataFrame, cfg: SimulationConfig
) -> pd.DataFrame:
    """Apply the current CRRA-sample rule even when estimates are cached."""
    estimates = estimates.copy()
    mask = estimates["success"].astype(bool) & np.isfinite(estimates["rho_hat"])
    if cfg.nlls_main_min_ccei is not None:
        mask &= estimates["ccei_i"] >= float(cfg.nlls_main_min_ccei)
    estimates["main_calibration_sample"] = mask
    return estimates


def summarize_crra_estimates(estimates: pd.DataFrame) -> pd.DataFrame:
    valid = estimates["success"] & np.isfinite(estimates["rho_hat"])
    samples = {
        "all_successful": valid,
        "ccei_ge_0.80": valid & (estimates["ccei_i"] >= 0.80),
        "ccei_ge_0.90": valid & (estimates["ccei_i"] >= 0.90),
        "main_calibration": estimates["main_calibration_sample"],
    }
    rows = []
    for label, mask in samples.items():
        values = estimates.loc[mask, "rho_hat"]
        rows.append(
            {
                "sample": label,
                "n": int(values.size),
                "mean": values.mean(),
                "sd": values.std(),
                "p05": values.quantile(0.05),
                "p10": values.quantile(0.10),
                "p25": values.quantile(0.25),
                "p50": values.quantile(0.50),
                "p75": values.quantile(0.75),
                "p90": values.quantile(0.90),
                "p95": values.quantile(0.95),
                "min": values.min(),
                "max": values.max(),
            }
        )
    return pd.DataFrame(rows)


def detailed_crra_distribution_statistics(estimates: pd.DataFrame) -> pd.DataFrame:
    """Report-ready CRRA distribution statistics in a tidy table."""
    valid = estimates["success"] & np.isfinite(estimates["rho_hat"])
    samples = {
        "all_successful": valid,
        "ccei_ge_0.80": valid & (estimates["ccei_i"] >= 0.80),
        "ccei_ge_0.90": valid & (estimates["ccei_i"] >= 0.90),
    }
    rows = []
    for sample, mask in samples.items():
        values = estimates.loc[mask, "rho_hat"]
        statistics = [
            ("N", np.nan, float(len(values))),
            ("Minimum", 0.0, float(values.min())),
        ]
        statistics.extend(
            (f"Q{q}", q / 100.0, float(values.quantile(q / 100.0)))
            for q in range(10, 100, 10)
        )
        statistics.extend(
            [
                ("Maximum", 1.0, float(values.max())),
                ("Count rho > 10", np.nan, float((values > 10).sum())),
                ("Share rho > 10", np.nan, float((values > 10).mean())),
                ("Count rho > 1e6", np.nan, float((values > 1e6).sum())),
                ("Share rho > 1e6", np.nan, float((values > 1e6).mean())),
                ("Count nonfinite", np.nan, float((~np.isfinite(values)).sum())),
            ]
        )
        for order, (statistic, percentile, value) in enumerate(statistics, start=1):
            rows.append(
                {
                    "sample": sample,
                    "order": order,
                    "statistic": statistic,
                    "percentile": percentile,
                    "value": value,
                }
            )
    return pd.DataFrame(rows)


def select_rho_scenarios(estimates: pd.DataFrame, cfg: SimulationConfig) -> pd.DataFrame:
    values = estimates.loc[estimates["main_calibration_sample"], "rho_hat"].dropna()
    if values.empty:
        raise ValueError("No valid CRRA estimates in the main calibration sample.")
    rows = []
    for name, (q_i, q_j) in cfg.heterogeneity_quantiles.items():
        rho_i = float(values.quantile(q_i))
        rho_j = float(values.quantile(q_j))
        if rho_i > rho_j:
            rho_i, rho_j = rho_j, rho_i
            q_i, q_j = q_j, q_i
        rows.append(
            {
                "heterogeneity": name,
                "quantile_i": q_i,
                "quantile_j": q_j,
                "rho_i": rho_i,
                "rho_j": rho_j,
                "rho_gap": rho_j - rho_i,
                "rho_midpoint": (rho_i + rho_j) / 2,
            }
        )
    return pd.DataFrame(rows)


def plot_crra_distribution(estimates: pd.DataFrame, scenarios: pd.DataFrame, output_file: Path) -> None:
    values = estimates.loc[estimates["main_calibration_sample"], "rho_hat"].dropna()
    display_cap = float(values.quantile(0.90))
    central = values.loc[values <= display_cap]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.hist(central, bins=35, color="#B7D5E1", edgecolor="white")
    quantile_lines = [
        (0.25, "#2A9D8F"),
        (0.50, "#1D4ED8"),
        (0.75, "#E76F51"),
    ]
    for quantile, color in quantile_lines:
        value = float(values.quantile(quantile))
        ax.axvline(
            value, color=color, linestyle="--", linewidth=2.0,
            label=f"Q{int(quantile * 100)} = {value:.3f}",
        )
    ax.set_xlabel(r"Estimated CRRA parameter $\hat{\rho}$")
    ax.set_ylabel("Count")
    ax.set_xlim(0, display_cap * 1.05)
    n = int(values.size)
    ax.set_title(f"Distribution of estimated CRRA parameters\n(N = {n:,} person-waves)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(output_file, dpi=300, facecolor="white")
    plt.close(fig)


def crra_distribution_comparison_statistics(estimates: pd.DataFrame) -> pd.DataFrame:
    """Side-by-side statistics for the full and CCEI >= 0.80 samples."""
    valid = estimates["success"] & np.isfinite(estimates["rho_hat"])
    samples = {
        "All successful estimates": estimates.loc[valid, "rho_hat"],
        "CCEI >= 0.80": estimates.loc[
            valid & (estimates["ccei_i"] >= 0.80), "rho_hat"
        ],
    }
    reported_quantiles = [10, 20, 25, 30, 40, 50, 60, 70, 75, 80, 90]
    statistic_functions = [
        ("N", lambda x: float(x.size)),
        ("Minimum", lambda x: float(x.min())),
        *[
            (f"Q{q}", lambda x, q=q: float(x.quantile(q / 100.0)))
            for q in reported_quantiles
        ],
        ("Maximum", lambda x: float(x.max())),
        ("Count rho > 10", lambda x: float((x > 10).sum())),
        ("Share rho > 10", lambda x: float((x > 10).mean())),
        ("Count rho > 1e6", lambda x: float((x > 1e6).sum())),
        ("Share rho > 1e6", lambda x: float((x > 1e6).mean())),
    ]
    rows = []
    for order, (statistic, function) in enumerate(statistic_functions, start=1):
        full_value = function(samples["All successful estimates"])
        filtered_value = function(samples["CCEI >= 0.80"])
        rows.append(
            {
                "order": order,
                "statistic": statistic,
                "all_successful": full_value,
                "ccei_ge_0.80": filtered_value,
                "difference_all_minus_filtered": full_value - filtered_value,
            }
        )
    return pd.DataFrame(rows)


def plot_crra_distribution_comparison(estimates: pd.DataFrame, output_dir: Path) -> None:
    """Create directly comparable full-sample and CCEI-filtered figures."""
    valid = estimates["success"] & np.isfinite(estimates["rho_hat"])
    samples = [
        (
            "all",
            "",
            estimates.loc[valid, "rho_hat"].dropna(),
        ),
        (
            "ccei_ge_080",
            "CCEI >= 0.80",
            estimates.loc[
                valid & (estimates["ccei_i"] >= 0.80), "rho_hat"
            ].dropna(),
        ),
    ]

    # Use the same horizontal scale and bin edges in both figures. Each sample
    # is displayed through its own Q90, matching the earlier report.
    common_cap = max(float(values.quantile(0.90)) for _, _, values in samples)
    bin_edges = np.linspace(0.0, common_cap, 36)
    x_max = common_cap * 1.05
    quantile_lines = [
        (0.25, "#2A9D8F"),
        (0.50, "#1D4ED8"),
        (0.75, "#E76F51"),
    ]

    for file_label, sample_label, values in samples:
        display_cap = float(values.quantile(0.90))
        central = values.loc[values <= display_cap]
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.hist(
            central,
            bins=bin_edges,
            color="#B7D5E1",
            edgecolor="white",
        )
        for quantile, color in quantile_lines:
            value = float(values.quantile(quantile))
            ax.axvline(
                value,
                color=color,
                linestyle="--",
                linewidth=2.0,
                label=f"Q{int(quantile * 100)} = {value:.3f}",
            )
        ax.set_xlabel(r"Estimated CRRA parameter $\hat{\rho}$")
        ax.set_ylabel("Count")
        ax.set_xlim(0, x_max)
        subtitle = (
            f"{sample_label} (N = {len(values):,}; upper 10% omitted)"
            if sample_label
            else f"(N = {len(values):,}; upper 10% omitted)"
        )
        ax.set_title(f"Distribution of estimated CRRA parameters\n{subtitle}")
        ax.legend(frameon=False)
        fig.tight_layout()
        fig.savefig(
            output_dir / f"crra_nlls_distribution_{file_label}.png",
            dpi=300,
            facecolor="white",
        )
        plt.close(fig)


# ---------------------------------------------------------------------------
# CRRA utility, stochastic choices, and gamma calibration
# ---------------------------------------------------------------------------


def crra_utility(consumption: np.ndarray, rho: float) -> np.ndarray:
    consumption = np.asarray(consumption, dtype=float)
    out = np.full_like(consumption, -np.inf, dtype=float)
    positive = consumption > 0
    if np.isclose(rho, 1.0):
        out[positive] = np.log(consumption[positive])
    else:
        out[positive] = consumption[positive] ** (1.0 - rho) / (1.0 - rho)
        if rho < 1.0:
            out[~positive] = 0.0
    return out


def normalized_state_utility(
    consumption: np.ndarray,
    rho: float,
    normalization: str,
    reference_consumption: float,
) -> np.ndarray:
    raw = crra_utility(consumption, rho)
    if normalization == "raw":
        return raw
    if normalization != "reference_marginal":
        raise ValueError(f"Unknown utility normalization: {normalization}")
    reference_level = crra_utility(np.array([reference_consumption]), rho)[0]
    reference_marginal = reference_consumption ** (-rho)
    return (raw - reference_level) / reference_marginal


def portfolio_grid(intercept_x: np.ndarray, intercept_y: np.ndarray, n_options: int) -> tuple[np.ndarray, np.ndarray]:
    share = np.linspace(0.0, 1.0, n_options)
    return intercept_x[:, None] * share, intercept_y[:, None] * (1.0 - share)


def expected_crra_grid(
    x: np.ndarray,
    y: np.ndarray,
    rho: float,
    normalization: str,
    reference_consumption: float,
) -> np.ndarray:
    return 0.5 * (
        normalized_state_utility(x, rho, normalization, reference_consumption)
        + normalized_state_utility(y, rho, normalization, reference_consumption)
    )


def softmax_probabilities(utility: np.ndarray, gamma: float) -> np.ndarray:
    utility = np.asarray(utility, dtype=float)
    row_max = np.max(utility, axis=1, keepdims=True)
    if np.any(~np.isfinite(row_max)):
        raise ValueError("At least one budget has no finite-utility option.")
    z = gamma * (utility - row_max)
    z[~np.isfinite(z)] = -np.inf
    exp_z = np.exp(np.clip(z, -745, 0))
    exp_z[~np.isfinite(z)] = 0.0
    denom = exp_z.sum(axis=1, keepdims=True)
    return exp_z / denom


def draw_choices_from_utility(utility: np.ndarray, gamma: float, rng: np.random.Generator) -> np.ndarray:
    probabilities = softmax_probabilities(utility, gamma)
    cdf = np.cumsum(probabilities, axis=1)
    draws = rng.random(len(utility))
    return np.minimum((cdf < draws[:, None]).sum(axis=1), utility.shape[1] - 1)


def sample_budget_arrays(pool: pd.DataFrame, n: int, replace: bool, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    if not replace and n > len(pool):
        raise ValueError("Cannot sample budgets without replacement: pool is too small.")
    idx = rng.choice(len(pool), size=n, replace=replace)
    sampled = pool.iloc[idx]
    return sampled["intercept_x"].to_numpy(float), sampled["intercept_y"].to_numpy(float)


def simulate_single_agent_dataset(
    pool: pd.DataFrame,
    rho: float,
    gamma: float,
    normalization: str,
    reference_consumption: float,
    cfg: SimulationConfig,
    rng: np.random.Generator,
) -> dict[str, np.ndarray]:
    ix, iy = sample_budget_arrays(pool, cfg.n_budgets, cfg.sample_budgets_with_replacement, rng)
    x, y = portfolio_grid(ix, iy, cfg.n_options)
    utility = expected_crra_grid(x, y, rho, normalization, reference_consumption)
    choice_idx = draw_choices_from_utility(utility, gamma, rng)
    rows = np.arange(cfg.n_budgets)
    return {"intercept_x": ix, "intercept_y": iy, "coord_x": x[rows, choice_idx], "coord_y": y[rows, choice_idx]}


def mean_simulated_ccei_for_gamma(
    gamma: float,
    rho: float,
    normalization: str,
    target_key: int,
    pool: pd.DataFrame,
    reference_consumption: float,
    cfg: SimulationConfig,
) -> float:
    values = []
    for rep in range(cfg.gamma_calibration_replications):
        rng = seed_rng(cfg.master_seed, 3100, target_key, rep)
        data = simulate_single_agent_dataset(
            pool, rho, gamma, normalization, reference_consumption, cfg, rng
        )
        values.append(ccei_dataset(data))
    return float(np.mean(values))


def calibrate_gamma_for_target(
    target_name: str,
    target_ccei: float,
    rho: float,
    normalization: str,
    target_key: int,
    pool: pd.DataFrame,
    reference_consumption: float,
    cfg: SimulationConfig,
) -> tuple[dict[str, Any], pd.DataFrame]:
    rows = []
    for gamma in cfg.gamma_grid:
        mean_ccei = mean_simulated_ccei_for_gamma(
            float(gamma), rho, normalization, target_key, pool, reference_consumption, cfg
        )
        rows.append(
            {
                "target_name": target_name,
                "normalization": normalization,
                "rho": rho,
                "gamma": float(gamma),
                "target_ccei": target_ccei,
                "mean_simulated_ccei": mean_ccei,
                "absolute_error": abs(mean_ccei - target_ccei),
            }
        )
    grid = pd.DataFrame(rows)
    best = grid.sort_values(["absolute_error", "gamma"]).iloc[0].to_dict()
    best["selected_gamma"] = best.pop("gamma")
    return best, grid


def mean_collective_ccei_for_gamma(
    gamma: float,
    rho_i: float,
    rho_j: float,
    alpha: float,
    normalization: str,
    target_key: int,
    pool: pd.DataFrame,
    reference_consumption: float,
    cfg: SimulationConfig,
) -> float:
    """Mean group CCEI from the actual alpha-weighted collective utility."""
    values = []
    for rep in range(cfg.gamma_calibration_replications):
        rng = seed_rng(cfg.master_seed, 4100, target_key, rep)
        ix, iy = sample_budget_arrays(
            pool, cfg.n_budgets, cfg.sample_budgets_with_replacement, rng
        )
        data = simulate_collective_dataset(
            ix, iy, rho_i, rho_j, alpha, gamma, normalization,
            reference_consumption, cfg, rng,
        )
        values.append(ccei_dataset(data))
    return float(np.mean(values))


def calibrate_collective_gamma_for_target(
    target_name: str,
    target_ccei: float,
    rho_i: float,
    rho_j: float,
    normalization: str,
    target_key: int,
    pool: pd.DataFrame,
    reference_consumption: float,
    cfg: SimulationConfig,
) -> tuple[dict[str, Any], pd.DataFrame]:
    rows = []
    for gamma in cfg.gamma_grid:
        mean_ccei = mean_collective_ccei_for_gamma(
            float(gamma), rho_i, rho_j, cfg.gamma_group_alpha_reference,
            normalization, target_key, pool, reference_consumption, cfg,
        )
        rows.append(
            {
                "target_name": target_name,
                "normalization": normalization,
                "rho_i": rho_i,
                "rho_j": rho_j,
                "alpha_reference": cfg.gamma_group_alpha_reference,
                "gamma": float(gamma),
                "target_ccei": target_ccei,
                "mean_simulated_ccei": mean_ccei,
                "absolute_error": abs(mean_ccei - target_ccei),
            }
        )
    grid = pd.DataFrame(rows)
    best = grid.sort_values(["absolute_error", "gamma"]).iloc[0].to_dict()
    best["selected_gamma"] = best.pop("gamma")
    return best, grid


def calibrate_all_gammas(
    rho_scenarios: pd.DataFrame,
    targets: dict[str, float],
    pool: pd.DataFrame,
    reference_consumption: float,
    cfg: SimulationConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calibrate Q25/Q50/Q75 precision levels from one curve per agent.

    Reusing the same simulated CCEI-versus-gamma curve for all three targets
    both saves computation and guarantees that low/median/high are selected
    from identical simulation draws.
    """
    best_rows = []
    grid_rows = []
    key = 0
    for norm in cfg.utility_normalizations:
        for scenario in rho_scenarios.itertuples(index=False):
            for member, rho in (("i", scenario.rho_i), ("j", scenario.rho_j)):
                key += 1
                curve_rows = []
                for gamma in cfg.gamma_grid:
                    mean_ccei = mean_simulated_ccei_for_gamma(
                        float(gamma), rho, norm, key, pool,
                        reference_consumption, cfg,
                    )
                    curve_rows.append(
                        {
                            "normalization": norm,
                            "rho": rho,
                            "gamma": float(gamma),
                            "mean_simulated_ccei": mean_ccei,
                        }
                    )
                curve = pd.DataFrame(curve_rows)
                for ccei_type in ("low", "median", "high"):
                    target = targets[f"individual_{ccei_type}"]
                    name = f"{scenario.heterogeneity}_{member}_{ccei_type}"
                    grid = curve.copy()
                    grid["target_name"] = name
                    grid["target_ccei"] = target
                    grid["absolute_error"] = abs(grid["mean_simulated_ccei"] - target)
                    best = grid.sort_values(["absolute_error", "gamma"]).iloc[0].to_dict()
                    best["selected_gamma"] = best.pop("gamma")
                    if best["absolute_error"] > cfg.gamma_calibration_tolerance:
                        raise ValueError(
                            f"Gamma calibration failed for {name}: target={target:.4f}, "
                            f"closest={best['mean_simulated_ccei']:.4f}, "
                            f"error={best['absolute_error']:.4f}. Expand gamma_grid."
                        )
                    best.update(
                        {
                            "heterogeneity": scenario.heterogeneity,
                            "agent": member,
                            "ccei_type": ccei_type,
                        }
                    )
                    grid["heterogeneity"] = scenario.heterogeneity
                    grid["agent"] = member
                    grid["ccei_type"] = ccei_type
                    best_rows.append(best)
                    grid_rows.append(grid)

            # The group also receives low/median/high precision calibrations,
            # targeted to the corresponding quantiles of empirical group CCEI.
            key += 1
            group_curve_rows = []
            for gamma in cfg.gamma_grid:
                mean_ccei = mean_collective_ccei_for_gamma(
                    float(gamma), scenario.rho_i, scenario.rho_j,
                    cfg.gamma_group_alpha_reference, norm, key, pool,
                    reference_consumption, cfg,
                )
                group_curve_rows.append(
                    {
                        "normalization": norm,
                        "rho_i": scenario.rho_i,
                        "rho_j": scenario.rho_j,
                        "alpha_reference": cfg.gamma_group_alpha_reference,
                        "gamma": float(gamma),
                        "mean_simulated_ccei": mean_ccei,
                    }
                )
            group_curve = pd.DataFrame(group_curve_rows)
            for ccei_type in ("low", "median", "high"):
                target = targets[f"group_{ccei_type}"]
                name = f"{scenario.heterogeneity}_group_{ccei_type}"
                grid = group_curve.copy()
                grid["target_name"] = name
                grid["target_ccei"] = target
                grid["absolute_error"] = abs(grid["mean_simulated_ccei"] - target)
                best = grid.sort_values(["absolute_error", "gamma"]).iloc[0].to_dict()
                best["selected_gamma"] = best.pop("gamma")
                if best["absolute_error"] > cfg.gamma_calibration_tolerance:
                    raise ValueError(
                        f"Gamma calibration failed for {name}: target={target:.4f}, "
                        f"closest={best['mean_simulated_ccei']:.4f}, "
                        f"error={best['absolute_error']:.4f}. Expand gamma_grid."
                    )
                best.update(
                    {
                        "heterogeneity": scenario.heterogeneity,
                        "agent": "g",
                        "ccei_type": ccei_type,
                    }
                )
                grid["heterogeneity"] = scenario.heterogeneity
                grid["agent"] = "g"
                grid["ccei_type"] = ccei_type
                best_rows.append(best)
                grid_rows.append(grid)

    return pd.DataFrame(best_rows), pd.concat(grid_rows, ignore_index=True)


def gamma_lookup(
    calibration: pd.DataFrame,
    normalization: str,
    heterogeneity: str,
    agent: str,
    ccei_type: str,
) -> float:
    row = calibration.loc[
        (calibration["normalization"] == normalization)
        & (calibration["heterogeneity"] == heterogeneity)
        & (calibration["agent"] == agent)
        & (calibration["ccei_type"] == ccei_type)
    ]
    if len(row) != 1:
        raise ValueError(
            f"Gamma lookup failed: norm={normalization}, heterogeneity={heterogeneity}, "
            f"agent={agent}, type={ccei_type}, matches={len(row)}"
        )
    return float(row["selected_gamma"].iloc[0])


# ---------------------------------------------------------------------------
# Main alpha-weighted collective-choice simulation
# ---------------------------------------------------------------------------


def parse_noise_scenario(name: str) -> tuple[str, str]:
    mapping = {
        "high_high": ("high", "high"),
        "high_low": ("high", "low"),
        "low_high": ("low", "high"),
        "low_low": ("low", "low"),
        "median_median": ("median", "median"),
    }
    if name not in mapping:
        raise ValueError(f"Unknown noise scenario: {name}")
    return mapping[name]


def simulate_collective_dataset(
    ix: np.ndarray,
    iy: np.ndarray,
    rho_i: float,
    rho_j: float,
    alpha: float,
    gamma_g: float,
    normalization: str,
    reference_consumption: float,
    cfg: SimulationConfig,
    rng: np.random.Generator,
) -> dict[str, np.ndarray]:
    x, y = portfolio_grid(ix, iy, cfg.n_options)
    utility_i = expected_crra_grid(x, y, rho_i, normalization, reference_consumption)
    utility_j = expected_crra_grid(x, y, rho_j, normalization, reference_consumption)
    # Avoid the indeterminate numerical expression 0 * (-inf) at alpha's
    # endpoints.  Economically, the zero-weight member drops out exactly.
    if np.isclose(alpha, 0.0):
        collective_utility = utility_j
    elif np.isclose(alpha, 1.0):
        collective_utility = utility_i
    else:
        collective_utility = alpha * utility_i + (1.0 - alpha) * utility_j
    choice_idx = draw_choices_from_utility(collective_utility, gamma_g, rng)
    rows = np.arange(cfg.n_budgets)
    return {"intercept_x": ix, "intercept_y": iy, "coord_x": x[rows, choice_idx], "coord_y": y[rows, choice_idx]}


def run_one_alpha_simulation(
    task_id: int,
    normalization: str,
    heterogeneity: str,
    rho_i: float,
    rho_j: float,
    noise_scenario: str,
    group_precision: str,
    alpha: float,
    replication: int,
    gamma_i: float,
    gamma_j: float,
    gamma_g: float,
    pool: pd.DataFrame,
    reference_consumption: float,
    cfg: SimulationConfig,
) -> dict[str, Any]:
    rng = seed_rng(cfg.master_seed, 9000, task_id, replication)

    data_i = simulate_single_agent_dataset(
        pool, rho_i, gamma_i, normalization, reference_consumption, cfg, rng
    )
    data_j = simulate_single_agent_dataset(
        pool, rho_j, gamma_j, normalization, reference_consumption, cfg, rng
    )
    ix_g, iy_g = sample_budget_arrays(
        pool, cfg.n_budgets, cfg.sample_budgets_with_replacement, rng
    )
    data_g = simulate_collective_dataset(
        ix_g, iy_g, rho_i, rho_j, alpha, gamma_g,
        normalization, reference_consumption, cfg, rng,
    )

    ex_i = ex_from_datasets(data_i, data_g)
    ex_j = ex_from_datasets(data_j, data_g)
    data_ij = stack_individual_datasets(data_i, data_j)
    ex_ij = ex_from_datasets(data_ij, data_g)
    ihat_i, ihat_j = ihat_from_ex(ex_i, ex_j, ex_ij)

    return {
        "normalization": normalization,
        "heterogeneity": heterogeneity,
        "noise_scenario": noise_scenario,
        "group_precision": group_precision,
        "true_alpha": alpha,
        "replication": replication,
        "rho_i": rho_i,
        "rho_j": rho_j,
        "rho_gap": rho_j - rho_i,
        "gamma_i": gamma_i,
        "gamma_j": gamma_j,
        "gamma_g": gamma_g,
        "Ihat_ig": ihat_i,
        "Ihat_jg": ihat_j,
        "alpha_hat": 1.0 - ihat_i if np.isfinite(ihat_i) else np.nan,
        "ex_ig": ex_i,
        "ex_jg": ex_j,
        "ex_Ng": ex_ij,
        "c_ig": 1.0 - ex_i,
        "c_jg": 1.0 - ex_j,
        "c_Ng": 1.0 - ex_ij,
        "ccei_i": ccei_dataset(data_i),
        "ccei_j": ccei_dataset(data_j),
        "ccei_g": ccei_dataset(data_g),
    }


def run_alpha_simulations(
    rho_scenarios: pd.DataFrame,
    gamma_calibration: pd.DataFrame,
    pool: pd.DataFrame,
    reference_consumption: float,
    cfg: SimulationConfig,
) -> pd.DataFrame:
    tasks = []
    task_id = 0
    for norm in cfg.utility_normalizations:
        for scenario in rho_scenarios.itertuples(index=False):
            for noise in cfg.noise_scenarios:
                type_i, type_j = parse_noise_scenario(noise)
                gamma_i = gamma_lookup(
                    gamma_calibration, norm, scenario.heterogeneity, "i", type_i
                )
                gamma_j = gamma_lookup(
                    gamma_calibration, norm, scenario.heterogeneity, "j", type_j
                )
                for group_precision in cfg.group_precision_scenarios:
                    gamma_g = gamma_lookup(
                        gamma_calibration, norm, scenario.heterogeneity, "g",
                        group_precision,
                    )
                    for alpha in cfg.alpha_grid:
                        task_id += 1
                        for rep in range(cfg.simulation_replications):
                            tasks.append(
                                (
                                    task_id, norm, scenario.heterogeneity,
                                    scenario.rho_i, scenario.rho_j, noise,
                                    group_precision, float(alpha), rep,
                                    gamma_i, gamma_j, gamma_g,
                                )
                            )

    results = Parallel(
        n_jobs=cfg.n_jobs,
        backend=cfg.backend,
        verbose=cfg.verbose,
    )(
        delayed(run_one_alpha_simulation)(
            task_id, norm, heterogeneity, rho_i, rho_j, noise,
            group_precision, alpha, rep,
            gamma_i, gamma_j, gamma_g, pool, reference_consumption, cfg,
        )
        for (
            task_id, norm, heterogeneity, rho_i, rho_j, noise,
            group_precision, alpha, rep,
            gamma_i, gamma_j, gamma_g,
        ) in tasks
    )
    return pd.DataFrame(results)


def summarize_simulations(results: pd.DataFrame) -> pd.DataFrame:
    keys = [
        "normalization", "heterogeneity", "noise_scenario",
        "group_precision", "true_alpha",
    ]
    rows = []
    for group_values, sub in results.groupby(keys, sort=True):
        defined = sub["alpha_hat"].dropna()
        row = dict(zip(keys, group_values))
        row.update(
            {
                "mean_alpha_hat": defined.mean(),
                "median_alpha_hat": defined.median(),
                "p05": defined.quantile(0.05),
                "p10": defined.quantile(0.10),
                "p90": defined.quantile(0.90),
                "p95": defined.quantile(0.95),
                "mean_bias": (defined - float(row["true_alpha"])).mean(),
                "rmse": np.sqrt(np.mean((defined - float(row["true_alpha"])) ** 2)),
                "n_runs": len(sub),
                "n_defined": len(defined),
                "n_undefined": int(sub["alpha_hat"].isna().sum()),
                "undefined_pct": 100.0 * sub["alpha_hat"].isna().mean(),
                "mean_ccei_i": sub["ccei_i"].mean(),
                "mean_ccei_j": sub["ccei_j"].mean(),
                "mean_ccei_g": sub["ccei_g"].mean(),
                "mean_c_Ng": sub["c_Ng"].mean(),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


def empirical_revealed_distance_targets(
    panel: pd.DataFrame, threshold: float
) -> pd.DataFrame:
    """Observed 1-Ihat for the empirically more rational member of each pair.

    The higher-CCEI indicator is constructed here from ccei_i and ccei_j.
    Under the paper's current definition, tied members are both selected.
    This gives the empirical analogue of member i in the mixed high/low
    simulation and avoids requiring a derived variable in the input file.
    """
    ensure_required_columns(
        panel, ["Ihat_ig", "ccei_i", "ccei_j"], "panel_individual"
    )
    ccei_i = pd.to_numeric(panel["ccei_i"], errors="coerce")
    ccei_j = pd.to_numeric(panel["ccei_j"], errors="coerce")
    high_ccei = ccei_i.notna() & ccei_j.notna() & (ccei_i >= ccei_j)
    chosen = panel.loc[high_ccei].copy()
    chosen["alpha_hat_observed"] = 1.0 - pd.to_numeric(
        chosen["Ihat_ig"], errors="coerce"
    )
    own_high = chosen["ccei_i"] >= threshold
    partner_high = chosen["ccei_j"] >= threshold
    chosen["noise_scenario"] = np.select(
        [own_high & partner_high, ~own_high & ~partner_high],
        ["high_high", "low_low"],
        default="high_low",
    )
    rows = []
    for noise, sub in chosen.groupby("noise_scenario"):
        values = sub["alpha_hat_observed"].dropna()
        rows.append(
            {
                "noise_scenario": noise,
                "n_pairs": len(sub),
                "n_defined": len(values),
                "undefined_pct": 100.0 * sub["alpha_hat_observed"].isna().mean(),
                "observed_mean_alpha_hat": values.mean(),
                "observed_median_alpha_hat": values.median(),
                "observed_p10": values.quantile(0.10),
                "observed_p90": values.quantile(0.90),
            }
        )
    return pd.DataFrame(rows)


def infer_alpha_from_empirical_distance(
    summary: pd.DataFrame, empirical_targets: pd.DataFrame
) -> pd.DataFrame:
    """Invert each simulated mean mapping to give a rough empirical alpha."""
    rows = []
    target_by_noise = empirical_targets.set_index("noise_scenario")
    for keys, sub in summary.groupby(
        ["normalization", "heterogeneity", "noise_scenario", "group_precision"],
        sort=True,
    ):
        norm, heterogeneity, noise, group_precision = keys
        if noise not in target_by_noise.index:
            continue
        sub = sub.sort_values("true_alpha")
        x = sub["true_alpha"].to_numpy(float)
        # Sampling noise can make the estimated mapping locally nonmonotone.
        # The cumulative maximum is a transparent monotone projection for a
        # descriptive inverse calibration, not an additional structural fit.
        y = np.maximum.accumulate(sub["mean_alpha_hat"].to_numpy(float))
        finite = np.isfinite(x) & np.isfinite(y)
        x, y = x[finite], y[finite]
        if len(x) < 2:
            continue
        y_unique, first = np.unique(y, return_index=True)
        x_unique = x[first]
        empirical = target_by_noise.loc[noise]
        row = {
            "normalization": norm,
            "heterogeneity": heterogeneity,
            "noise_scenario": noise,
            "group_precision": group_precision,
            "n_empirical_pairs": int(empirical["n_pairs"]),
            "mapping_min": float(y_unique.min()),
            "mapping_max": float(y_unique.max()),
        }
        for statistic in ("mean", "median"):
            target = float(empirical[f"observed_{statistic}_alpha_hat"])
            row[f"observed_{statistic}_alpha_hat"] = target
            row[f"implied_alpha_from_{statistic}"] = float(
                np.interp(target, y_unique, x_unique)
            )
            row[f"{statistic}_outside_simulated_range"] = bool(
                target < y_unique.min() or target > y_unique.max()
            )
        rows.append(row)
    return pd.DataFrame(rows)


def plot_alpha_mappings(summary: pd.DataFrame, output_dir: Path) -> None:
    """Reproduce the original Figure A7 visual design for every setting.

    The reporting figures use the original CRRA cardinalization only.  Each
    figure contains one heterogeneity case, so its visual elements match A7:
    blue mean dots, nested sky-blue 5--95 and 10--90 percent bands, and a red
    dashed 45-degree line.
    """
    reporting = summary.loc[
        (summary["normalization"] == "raw")
        & summary["noise_scenario"].isin(["high_high", "high_low", "low_low"])
    ].copy()
    figure_dir = output_dir / "figures_main_style"
    figure_dir.mkdir(parents=True, exist_ok=True)

    for (heterogeneity, noise, group_precision), sub in reporting.groupby(
        ["heterogeneity", "noise_scenario", "group_precision"], sort=True
    ):
        sub = sub.sort_values("true_alpha")
        x = sub["true_alpha"].to_numpy(float)
        heterogeneity_label = heterogeneity.title()
        individual_label = {
            "high_high": "High–High",
            "high_low": "High–Low",
            "low_low": "Low–Low",
        }[noise]
        group_label = group_precision.title()
        setting_label = (
            f"Preference heterogeneity: {heterogeneity_label}  |  "
            f"Individual precision: {individual_label}  |  "
            f"Group precision: {group_label}"
        )

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.fill_between(
            x, sub["p05"], sub["p95"], color="skyblue", alpha=0.2,
            label="5–95%",
        )
        ax.fill_between(
            x, sub["p10"], sub["p90"], color="skyblue", alpha=0.4,
            label="10–90%",
        )
        ax.scatter(x, sub["mean_alpha_hat"], color="blue", label="Mean (1 - Ihat_ig)")
        ax.plot(x, x, "r--", label="y=x")
        ax.set(
            xlabel="True alpha (weight on i)",
            ylabel="1 - Ihat_ig simulated",
        )
        ax.set_title(
            "True alpha vs cross-partition (1 - Ihat_ig)\n" + setting_label,
            pad=10,
        )
        ax.set_xticks(x)
        ax.grid(True)
        ax.legend(loc="upper left")
        fig.subplots_adjust(bottom=0.25)

        for row in sub.itertuples(index=False):
            ax.text(
                row.true_alpha, -0.12,
                f"{row.n_undefined}/{row.n_runs}\n({row.undefined_pct:.1f}%)",
                transform=ax.get_xaxis_transform(), ha="center", va="top",
                color="red", fontsize=7,
            )
        ax.text(
            0.5, -0.21, "Undefined count/percent per alpha",
            transform=ax.transAxes, ha="center", va="top", color="red", fontsize=9,
        )

        filename = (
            f"alpha_mapping_{heterogeneity}_{noise}_group_{group_precision}.png"
        )
        fig.savefig(figure_dir / filename, dpi=300, facecolor="white")
        plt.close(fig)


def plot_utility_normalization_comparison(summary: pd.DataFrame, output_file: Path) -> None:
    if summary["normalization"].nunique() < 2:
        return
    for group_precision, group_data in summary.groupby("group_precision"):
        fig, axes = plt.subplots(
            len(group_data["noise_scenario"].unique()),
            len(group_data["heterogeneity"].unique()),
            figsize=(13, 10), sharex=True, sharey=True,
        )
        axes = np.atleast_2d(axes)
        noises = list(group_data["noise_scenario"].drop_duplicates())
        heterogeneities = list(group_data["heterogeneity"].drop_duplicates())
        line_styles = {"raw": "-", "reference_marginal": "--"}
        for row_idx, noise in enumerate(noises):
            for col_idx, heterogeneity in enumerate(heterogeneities):
                ax = axes[row_idx, col_idx]
                sub = group_data.loc[
                    (group_data["noise_scenario"] == noise)
                    & (group_data["heterogeneity"] == heterogeneity)
                ]
                for norm, line in sub.groupby("normalization"):
                    line = line.sort_values("true_alpha")
                    ax.plot(
                        line["true_alpha"], line["mean_alpha_hat"],
                        linestyle=line_styles.get(norm, "-"), marker="o", markersize=3,
                        label=norm.replace("_", " "),
                    )
                ax.plot([0, 1], [0, 1], color="grey", linestyle=":", linewidth=1)
                ax.set_title(f"{noise.replace('_', '-')} / {heterogeneity}")
                ax.grid(alpha=0.2)
                if row_idx == len(noises) - 1:
                    ax.set_xlabel(r"$\alpha$")
                if col_idx == 0:
                    ax.set_ylabel(r"Mean $1-\widehat{I}_{ig}$")
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
        fig.suptitle(
            f"Utility normalization sensitivity; group {group_precision} precision",
            y=0.995,
        )
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        group_file = output_file.with_name(
            f"{output_file.stem}_group_{group_precision}{output_file.suffix}"
        )
        fig.savefig(group_file, dpi=300, facecolor="white")
        plt.close(fig)


# ---------------------------------------------------------------------------
# Validation and pipeline
# ---------------------------------------------------------------------------


def run_self_checks() -> None:
    rng = np.random.default_rng(20260823)
    for _ in range(100):
        p = rng.uniform(0.4, 2.0, (2, 2))
        x = rng.uniform(0.1, 2.0, (2, 2))
        ex = ex_cross(p, x, np.array([0, 1]))
        identity = min(
            ccei_garp(p[:, :1], x[:, :1]),
            ccei_garp(p[:, 1:], x[:, 1:]),
            ex,
        )
        if not np.isclose(ccei_garp(p, x), identity, atol=2e-6):
            raise AssertionError("Cross-CCEI decomposition self-check failed.")
    i_hat, j_hat = ihat_from_ex(0.8, 0.6, 0.5)
    if not np.isclose(i_hat + j_hat, 1.0):
        raise AssertionError("Ihat adding-up self-check failed.")

    lpr = np.array([-1.0, 0.0, 1.0])
    pred = eut_crra_predicted_log_demand_ratio(lpr, rho=0.5, w=0.001)
    if not np.allclose(pred, np.array([2.0, 0.0, -2.0])):
        raise AssertionError("Constrained EUT-CRRA demand self-check failed.")
    print("Self-checks: PASS")


def apply_smoke_test_overrides(cfg: SimulationConfig) -> None:
    cfg.gamma_grid = list(np.geomspace(1e-4, 1e14, 10))
    cfg.gamma_calibration_replications = 4
    cfg.gamma_calibration_tolerance = 0.25
    cfg.simulation_replications = 3
    cfg.alpha_grid = [0.0, 0.5, 1.0]
    cfg.utility_normalizations = ["raw"]
    cfg.n_jobs = 1
    cfg.verbose = 0


def run_pipeline(
    cfg: SimulationConfig,
    stage: str,
    force: bool = False,
    force_estimation: bool | None = None,
    force_downstream: bool | None = None,
) -> None:
    # Separate switches are useful for the two Jupyter-facing driver files:
    # part 2 may be recomputed without repeating the NLLS work from part 1.
    if force_estimation is None:
        force_estimation = force
    if force_downstream is None:
        force_downstream = force
    start = time.time()
    data_dir = Path(cfg.data_dir).expanduser().resolve()
    output_dir = Path(cfg.output_dir).expanduser().resolve()
    output_dir.mkdir(parents=True, exist_ok=True)
    save_json(output_dir / "config.json", asdict(cfg))

    base_raw = prepare_raw(data_dir / cfg.base_raw_file, post=0)
    end_raw = prepare_raw(data_dir / cfg.end_raw_file, post=1)
    panel = load_panel_individual(data_dir / cfg.panel_individual_file)
    raw_frames = [base_raw, end_raw]

    if cfg.budget_pool == "all":
        pool = canonical_budget_pool(raw_frames)
    elif cfg.budget_pool == "rational_legacy":
        pool = legacy_rational_budget_pool(data_dir / cfg.legacy_budget_file)
    else:
        raise ValueError("budget_pool must be 'all' or 'rational_legacy'.")
    pool_summary = {
        "budget_pool": cfg.budget_pool,
        "n_rows": len(pool),
        "n_unique_budgets": int(pool[["intercept_x", "intercept_y"]].drop_duplicates().shape[0]),
        "intercept_x_min": float(pool["intercept_x"].min()),
        "intercept_x_max": float(pool["intercept_x"].max()),
        "intercept_y_min": float(pool["intercept_y"].min()),
        "intercept_y_max": float(pool["intercept_y"].max()),
    }
    save_json(output_dir / "budget_pool_summary.json", pool_summary)

    reference_consumption = (
        float(cfg.reference_consumption)
        if cfg.reference_consumption is not None
        else float(pd.concat([pool["intercept_x"], pool["intercept_y"]]).median() / 2.0)
    )
    save_json(output_dir / "utility_reference.json", {"reference_consumption": reference_consumption})

    estimates_path = output_dir / "crra_nlls_person_wave.csv"
    if force_estimation or not estimates_path.exists():
        estimates = estimate_crra_distribution(raw_frames, panel, cfg)
        estimates.to_csv(estimates_path, index=False)
    else:
        estimates = pd.read_csv(estimates_path, dtype={"id": str})
    # The CRRA estimates themselves do not change when the calibration sample
    # changes, so refresh this flag even when the expensive NLLS output is read
    # from disk.
    estimates = apply_main_calibration_sample(estimates, cfg)
    estimates.to_csv(estimates_path, index=False)
    crra_summary = summarize_crra_estimates(estimates)
    crra_summary.to_csv(output_dir / "crra_nlls_summary.csv", index=False)
    detailed_crra_distribution_statistics(estimates).to_csv(
        output_dir / "crra_distribution_statistics.csv", index=False
    )
    rho_scenarios = select_rho_scenarios(estimates, cfg)
    rho_scenarios.to_csv(output_dir / "rho_heterogeneity_scenarios.csv", index=False)
    plot_crra_distribution(estimates, rho_scenarios, output_dir / "crra_nlls_distribution.png")
    crra_distribution_comparison_statistics(estimates).to_csv(
        output_dir / "crra_distribution_comparison.csv", index=False
    )
    plot_crra_distribution_comparison(estimates, output_dir)

    targets = empirical_ccei_targets(panel, cfg.high_ccei_threshold)
    save_json(output_dir / "empirical_ccei_targets.json", targets)

    if stage == "estimate":
        print(f"Estimation outputs saved to {output_dir}")
        return

    gamma_path = output_dir / "gamma_calibration_selected.csv"
    gamma_grid_path = output_dir / "gamma_calibration_grid.csv"
    if force_downstream or not gamma_path.exists() or not gamma_grid_path.exists():
        gamma_selected, gamma_grid = calibrate_all_gammas(
            rho_scenarios, targets, pool, reference_consumption, cfg
        )
        gamma_selected.to_csv(gamma_path, index=False)
        gamma_grid.to_csv(gamma_grid_path, index=False)
    else:
        gamma_selected = pd.read_csv(gamma_path)

    if stage == "calibrate":
        print(f"Calibration outputs saved to {output_dir}")
        return

    results_path = output_dir / "alpha_simulation_results.csv"
    if force_downstream or not results_path.exists():
        results = run_alpha_simulations(
            rho_scenarios, gamma_selected, pool, reference_consumption, cfg
        )
        results.to_csv(results_path, index=False)
    else:
        results = pd.read_csv(results_path)
    summary = summarize_simulations(results)
    summary.to_csv(output_dir / "alpha_simulation_summary.csv", index=False)
    empirical_distance = empirical_revealed_distance_targets(
        panel, cfg.high_ccei_threshold
    )
    empirical_distance.to_csv(
        output_dir / "empirical_revealed_distance_targets.csv", index=False
    )
    implied_alpha = infer_alpha_from_empirical_distance(summary, empirical_distance)
    implied_alpha.to_csv(
        output_dir / "implied_alpha_from_empirical_distance.csv", index=False
    )
    plot_alpha_mappings(summary, output_dir)
    plot_utility_normalization_comparison(
        summary, output_dir / "alpha_mapping_utility_normalization_comparison.png"
    )

    diagnostics = {
        "elapsed_seconds": time.time() - start,
        "n_raw_simulation_rows": len(results),
        "overall_undefined_count": int(results["alpha_hat"].isna().sum()),
        "overall_undefined_pct": 100.0 * float(results["alpha_hat"].isna().mean()),
        "mean_simulated_ccei_i": float(results["ccei_i"].mean()),
        "mean_simulated_ccei_j": float(results["ccei_j"].mean()),
        "mean_simulated_ccei_g": float(results["ccei_g"].mean()),
    }
    save_json(output_dir / "run_diagnostics.json", diagnostics)
    print(json.dumps(diagnostics, indent=2))
    print(f"All outputs saved to {output_dir}")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--stage",
        choices=["estimate", "calibrate", "all"],
        default="all",
        help="Stop after CRRA estimation, gamma calibration, or run everything.",
    )
    parser.add_argument("--smoke-test", action="store_true", help="Run a very small end-to-end test.")
    parser.add_argument("--force", action="store_true", help="Recompute cached outputs.")
    parser.add_argument("--output-dir", type=str, default=None, help="Override the output directory.")
    parser.add_argument(
        "--budget-pool",
        choices=["all", "rational_legacy"],
        default=None,
        help="Use the full empirical pool or the legacy rational-sample pool.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    cfg = SimulationConfig()
    if args.output_dir:
        cfg.output_dir = args.output_dir
    if args.budget_pool:
        cfg.budget_pool = args.budget_pool
    if args.smoke_test:
        apply_smoke_test_overrides(cfg)
        cfg.output_dir = str(Path(cfg.output_dir) / "smoke_test")
    run_self_checks()
    run_pipeline(cfg, stage=args.stage, force=args.force)



## 1. Replication-package paths and empirical CCEI check

In [3]:
from IPython.display import display

PACKAGE_DIR = Path.cwd().resolve()
if not (PACKAGE_DIR / 'data' / 'panel_individual.dta').exists():
    raise FileNotFoundError(
        'Open this notebook from the Code directory. '
        f'Current working directory: {PACKAGE_DIR}'
    )

DATA_DIR = PACKAGE_DIR / 'data'
OUTPUT_DIR = PACKAGE_DIR / 'results' / 'alpha_weighted_simulation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

panel_check = pd.read_stata(
    DATA_DIR / 'panel_individual.dta', convert_categoricals=False
)
individual_ccei = (
    panel_check.drop_duplicates(['id', 'post'])['ccei_i'].dropna()
)
group_ccei = (
    panel_check.drop_duplicates(['group_id', 'post'])['ccei_g'].dropna()
)

def ccei_summary(values, sample):
    return {
        'sample': sample, 'N': len(values), 'mean': values.mean(),
        'sd': values.std(), 'min': values.min(),
        'q10': values.quantile(.10), 'q25': values.quantile(.25),
        'q50': values.quantile(.50), 'q75': values.quantile(.75),
        'q90': values.quantile(.90), 'max': values.max(),
    }

ccei_check = pd.DataFrame([
    ccei_summary(individual_ccei, 'Individual person-wave'),
    ccei_summary(group_ccei, 'Group pair-wave'),
])
display(ccei_check.round(6))
ccei_check.to_csv(OUTPUT_DIR / 'empirical_ccei_distribution.csv', index=False)

# These checks identify an unintended change in the packaged panel before
# the time-consuming simulation begins.
assert len(individual_ccei) == 2608
assert len(group_ccei) == 1304
assert np.isclose(individual_ccei.quantile(.25), 0.8723425865, atol=1e-8)
assert np.isclose(individual_ccei.quantile(.50), 0.9806590080, atol=1e-8)
assert np.isclose(individual_ccei.quantile(.75), 1.0, atol=1e-8)
assert np.isclose(group_ccei.quantile(.25), 0.9045834541, atol=1e-8)
assert np.isclose(group_ccei.quantile(.50), 0.9945955276, atol=1e-8)
assert np.isclose(group_ccei.quantile(.75), 1.0, atol=1e-8)
print('Empirical CCEI check: PASS')

,sample,N,mean,sd,min,q10,q25,q50,q75,q90,max
0,Individual person-wave,2608,0.916218,0.127679,0.208394,0.750222,0.872343,0.980659,1.0,1.0,1.0
1,Group pair-wave,1304,0.923993,0.134916,0.161963,0.748035,0.904583,0.994596,1.0,1.0,1.0


Empirical CCEI check: PASS


## 2. Simulation settings

In [5]:
CONFIG = SimulationConfig(
    data_dir=str(DATA_DIR),
    output_dir=str(OUTPUT_DIR),
    n_budgets=18,
    n_options=200,
    budget_pool='all',
    sample_budgets_with_replacement=False,
    boundary_w=0.001,
    min_nlls_observations=12,
    nlls_main_min_ccei=None,
    heterogeneity_quantiles={
        'small': (0.40, 0.60),
        'medium': (0.25, 0.75),
        'large': (0.10, 0.90),
    },
    gamma_calibration_replications=150,
    gamma_calibration_tolerance=0.03,
    gamma_group_alpha_reference=0.50,
    alpha_grid=[round(x / 10, 1) for x in range(11)],
    simulation_replications=100,
    noise_scenarios=['high_high', 'high_low', 'low_low'],
    group_precision_scenarios=['low', 'median', 'high'],
    utility_normalizations=['raw'],
    master_seed=20260823,
    n_jobs=-1,
    backend='loky',
    verbose=5,
)

display(pd.Series(asdict(CONFIG), name='value'))

data_dir                                  C:\Users\hahn0\RP\Replication_Package\data
output_dir                         C:\Users\hahn0\RP\Replication_Package\results\...
base_raw_file                                                           base_raw.dta
end_raw_file                                                             end_raw.dta
panel_individual_file                                           panel_individual.dta
n_budgets                                                                         18
n_options                                                                        200
budget_pool                                                                      all
legacy_budget_file                                        baseline_raw_129groups.csv
sample_budgets_with_replacement                                                False
boundary_w                                                                     0.001
nlls_main_min_ccei                                               

## 3. Run

This cell reuses completed output files when they exist. On a fresh run, it estimates CRRA, calibrates gamma, and runs all 29,700 simulation replications.

In [10]:
run_self_checks()
run_pipeline(
    CONFIG,
    stage='all',
    force_estimation=False,
    force_downstream=False,
)

Self-checks: PASS
{
  "elapsed_seconds": 10.94881796836853,
  "n_raw_simulation_rows": 29700,
  "overall_undefined_count": 1050,
  "overall_undefined_pct": 3.535353535353535,
  "mean_simulated_ccei_i": 0.9587304101460566,
  "mean_simulated_ccei_j": 0.9150604910266477,
  "mean_simulated_ccei_g": 0.954428793660354
}
All outputs saved to C:\Users\hahn0\RP\Replication_Package\results\alpha_weighted_simulation


## 4. Main outputs

In [12]:
rho_scenarios = pd.read_csv(OUTPUT_DIR / 'rho_heterogeneity_scenarios.csv')
gamma_selected = pd.read_csv(OUTPUT_DIR / 'gamma_calibration_selected.csv')
simulation_summary = pd.read_csv(OUTPUT_DIR / 'alpha_simulation_summary.csv')

print('Output directory:', OUTPUT_DIR)
display(rho_scenarios)
display(gamma_selected[[
    'heterogeneity', 'agent', 'ccei_type', 'target_ccei',
    'selected_gamma', 'mean_simulated_ccei', 'absolute_error'
]])
display(simulation_summary.head())
print('Main-style figures:', OUTPUT_DIR / 'figures_main_style')

Output directory: C:\Users\hahn0\RP\Replication_Package\results\alpha_weighted_simulation


,heterogeneity,quantile_i,quantile_j,rho_i,rho_j,rho_gap,rho_midpoint
0,small,0.40,0.60,0.405598,0.699213,0.293616,0.552405
1,medium,0.25,0.75,0.235870,1.184313,0.948442,0.710092
2,large,0.10,0.90,0.054835,4.248728,4.193893,2.151782


,heterogeneity,agent,ccei_type,target_ccei,selected_gamma,mean_simulated_ccei,absolute_error
0,small,i,low,0.872343,4.641589e-02,0.896041,0.023698
1,small,i,median,0.980659,2.154435e-01,0.983092,0.002433
2,small,i,high,1.000000,1.000000e+01,1.000000,0.000000
3,small,j,low,0.872343,2.154435e-01,0.855698,0.016644
4,small,j,median,0.980659,2.154435e+00,0.987602,0.006943
5,small,j,high,1.000000,4.641589e+02,1.000000,0.000000
6,small,g,low,0.904583,1.000000e-01,0.889345,0.015238
7,small,g,median,0.994596,1.000000e+00,0.996867,0.002271
8,small,g,high,1.000000,1.000000e+01,1.000000,0.000000
9,medium,i,low,0.872343,1.000000e-02,0.853590,0.018752


,normalization,heterogeneity,noise_scenario,group_precision,true_alpha,mean_alpha_hat,median_alpha_hat,p05,p10,p90,...,mean_bias,rmse,n_runs,n_defined,n_undefined,undefined_pct,mean_ccei_i,mean_ccei_j,mean_ccei_g,mean_c_Ng
0,raw,large,high_high,high,0.0,0.525690,0.526929,0.220313,0.285686,0.749478,...,0.525690,0.553682,100,100,0,0.0,0.999991,0.999998,0.769238,0.284138
1,raw,large,high_high,high,0.1,0.994986,1.000000,0.993086,0.999787,1.000000,...,0.894986,0.895350,100,100,0,0.0,1.000000,1.000000,1.000000,0.258330
2,raw,large,high_high,high,0.2,0.997057,1.000000,0.997464,0.999027,1.000000,...,0.797057,0.797181,100,100,0,0.0,0.999994,1.000000,1.000000,0.266165
3,raw,large,high_high,high,0.3,0.995856,1.000000,0.995581,1.000000,1.000000,...,0.695856,0.696169,100,100,0,0.0,0.999997,1.000000,1.000000,0.261218
4,raw,large,high_high,high,0.4,0.993855,1.000000,0.998682,1.000000,1.000000,...,0.593855,0.594970,100,100,0,0.0,1.000000,1.000000,1.000000,0.257497


Main-style figures: C:\Users\hahn0\RP\Replication_Package\results\alpha_weighted_simulation\figures_main_style
